# Exp 4 — Final Model

**Full configuration: SPT + LSA + Hard Distillation with Distillation Token**

- Shifted Patch Tokenization (SPT): 15-channel patch embedding (3 orig + 4x 3 shifts)
- Locality Self-Attention (LSA): diagonal masking + learnable temperature
- Hard distillation + distillation token: 0.5·CE(cls, y) + 0.5·CE(dist, argmax(teacher))
- Teacher: ResNet-18 at project2/checkpoints/run10_final_9k/best_model.pt (95% val acc)

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL = "YOUR_GIT_REMOTE_HERE"   # TODO: set your git remote URL
REPO_DIR = "/content/repo"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "torchvision", "huggingface-hub",
                    "matplotlib", "seaborn", "scikit-learn"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(".", "..", ".."))

P3_SRC = os.path.join(REPO_DIR, "project3", "src")
P2_SRC = os.path.join(REPO_DIR, "project2", "src")
for p in [P3_SRC, P2_SRC]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(P3_SRC)

In [ ]:
# ── Data + device ────────────────────────────────────────────────────────────
import torch, numpy as np

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/cs148/data/dataset"  # TODO: adjust
else:
    DATA_DIR = os.path.join(REPO_DIR, "project2", "data", "dataset")

TEACHER_PATH = os.path.join(REPO_DIR, "project2", "checkpoints", "run10_final_9k", "best_model.pt")
SAVE_DIR = "/content/checkpoints/exp4_final" if IN_COLAB else "../checkpoints/exp4_final"

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ── Sanity check: model with all features ────────────────────────────────────
from model import build_model, count_parameters

model = build_model(use_spt=True, use_lsa=True, use_dist_token=True)
n = count_parameters(model)
print(f"ViT (SPT+LSA+dist token): {n:,} params ({n/1e6:.2f}M)")

x = torch.randn(2, 3, 128, 128)
inf_out = model(x)                    # inference: single tensor
cls_out, dist_out = model.forward_train(x)  # training: both heads
print(f"Inference output: {inf_out.shape}")
print(f"Training: cls={cls_out.shape}, dist={dist_out.shape}")

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
import argparse

args = argparse.Namespace(
    data_dir=DATA_DIR,
    save_dir=SAVE_DIR,
    # Training
    epochs=100,
    batch_size=128,
    lr=3e-4,
    min_lr=1e-5,
    weight_decay=0.1,
    warmup_epochs=5,
    drop_path_rate=0.2,
    img_size=128,
    repeat_aug=2,
    patience=0,
    num_workers=2,
    seed=42,
    # Dataset
    synthetic_n=0,
    val_fraction=0.15,
    # Model: SPT + LSA + hard distillation with distillation token
    use_spt=True,
    use_lsa=True,
    distillation="hard-dist",
    teacher_path=TEACHER_PATH,
    tau=4.0,
    lambda_dist=0.5,
    resume_checkpoint=None,
)

torch.manual_seed(args.seed)
np.random.seed(args.seed)
print("Config ready.")

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
import train
train.train(args)

In [ ]:
# ── Export pipeline for HF submission ────────────────────────────────────────
from pipeline import load_pipeline

best_ckpt = os.path.join(SAVE_DIR, "best_model.pt")
pl = load_pipeline(best_ckpt, device="cpu")
pl.save_pipeline_local("pipeline-vit.pt")
print("Saved pipeline-vit.pt")

# Test it
scripted = torch.jit.load("pipeline-vit.pt", map_location="cpu")
scripted.eval()
dummy = torch.randn(4, 3, 128, 128)
preds = scripted(dummy)
print(f"Pipeline test: input {dummy.shape} -> predictions {preds.tolist()}")

In [ ]:
# ── (Optional) Push to HuggingFace ──────────────────────────────────────────
# Fill in your token to push
HF_TOKEN = ""  # TODO: set your HF write token

if HF_TOKEN:
    pl.push_to_hub(
        token=HF_TOKEN,
        hf_info={
            "username": "arjuns07",
            "repo_name": "ee148a-project",
            "filename": "pipeline-vit.pt",
            "token": HF_TOKEN,
        }
    )
else:
    print("HF_TOKEN not set — skipping push.")

In [ ]:
# ── Download results (Colab only) ────────────────────────────────────────────
if IN_COLAB:
    import shutil
    from google.colab import files
    zip_path = "/content/exp4_final_results.zip"
    shutil.make_archive("/content/exp4_final_results", "zip", SAVE_DIR)
    files.download(zip_path)
    files.download("pipeline-vit.pt")